In [1]:
# 1) Imports + config

from pathlib import Path
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# 2) Load processed dataset v2

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

processed_path = DATA_PROCESSED / "jailbreak_benchmarks_processed_v2.csv"
df = pd.read_csv(processed_path)

print("Rows:", len(df))
print("\nSplit counts:")
print(df["split"].value_counts())
print("\nLabel counts:")
print(df["label"].value_counts())

Rows: 1647

Split counts:
split
ood_test    768
train       615
val         132
test        132
Name: count, dtype: int64

Label counts:
label
1    993
0    654
Name: count, dtype: int64


In [3]:
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()
df_ood   = df[df["split"] == "ood_test"].copy()

for name, d in [("train", df_train), ("val", df_val), ("test", df_test), ("ood_test", df_ood)]:
    print(f"{name:8s}", d.shape, d["label"].value_counts().to_dict())


train    (615, 8) {1: 426, 0: 189}
val      (132, 8) {1: 92, 0: 40}
test     (132, 8) {1: 91, 0: 41}
ood_test (768, 8) {1: 384, 0: 384}


In [4]:
X_train_text = df_train["prompt_text"].astype(str).tolist()
X_val_text   = df_val["prompt_text"].astype(str).tolist()
X_test_text  = df_test["prompt_text"].astype(str).tolist()
X_ood_text   = df_ood["prompt_text"].astype(str).tolist()

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values


In [5]:
# 3) Embeddings: Compute

sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def encode_texts(texts, batch_size=32):
    return sem_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

X_sem_train = encode_texts(X_train_text)
X_sem_val   = encode_texts(X_val_text)
X_sem_test  = encode_texts(X_test_text)
X_sem_ood   = encode_texts(X_ood_text)

print("\nEmbedding shapes:")
print("Train:", X_sem_train.shape)
print("Val:  ", X_sem_val.shape)
print("Test: ", X_sem_test.shape)
print("OOD:  ", X_sem_ood.shape)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]


Embedding shapes:
Train: (615, 384)
Val:   (132, 384)
Test:  (132, 384)
OOD:   (768, 384)


In [6]:
# 4) Train semantic-only classifier

clf = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

clf.fit(X_sem_train, y_train)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

LogisticRegression(class_weight='balanced', max_iter=5000, n_jobs=-1,
                   random_state=42)

In [7]:
# 5) Evaluate

def eval_split(name, X, y):
    y_pred = clf.predict(X)
    acc = accuracy_score(y, y_pred)
    macro = f1_score(y, y_pred, average="macro")
    print(f"\n=== {name} ===")
    print(classification_report(y, y_pred, digits=3))
    return {"split": name, "accuracy": float(acc), "macro_f1": float(macro)}

val_m  = eval_split("VAL",  X_sem_val,  y_val)
test_m = eval_split("TEST", X_sem_test, y_test)
ood_m  = eval_split("OOD",  X_sem_ood,  y_ood)

summary = pd.DataFrame([val_m, test_m, ood_m])
summary["model"] = "SEM_LOGREG"
summary = summary[["model", "split", "accuracy", "macro_f1"]]

print("\n=== Semantic Baseline Summary (v2) ===")
display(summary)


=== VAL ===
              precision    recall  f1-score   support

           0      0.947     0.900     0.923        40
           1      0.957     0.978     0.968        92

    accuracy                          0.955       132
   macro avg      0.952     0.939     0.945       132
weighted avg      0.954     0.955     0.954       132


=== TEST ===
              precision    recall  f1-score   support

           0      0.868     0.805     0.835        41
           1      0.915     0.945     0.930        91

    accuracy                          0.902       132
   macro avg      0.892     0.875     0.883       132
weighted avg      0.900     0.902     0.900       132


=== OOD ===
              precision    recall  f1-score   support

           0      0.659     0.906     0.763       384
           1      0.850     0.531     0.654       384

    accuracy                          0.719       768
   macro avg      0.755     0.719     0.709       768
weighted avg      0.755     0.719 

,model,split,accuracy,macro_f1
0,SEM_LOGREG,VAL,0.954545,0.945409
1,SEM_LOGREG,TEST,0.901515,0.882586
2,SEM_LOGREG,OOD,0.718750,0.708502
